**Changed on branch `feature/downtime-drivers`:** the `tkinter` file picker was replaced by `data_utils.load_*()`, which reads the parquet file from the repository root, so the notebook runs without clicking and is reproducible. `inplace=True` was removed (deprecated in pandas 3). `failure_comments` is no longer dropped: a new cell extracts `failure_depth_m` and `failure_rod_number` from it and adds standardised `failure_type_std` / `cause_std`. `pd.factorize` was replaced by `category` dtype (factorize turned missing values into -1).

In [1]:
import pandas as pd

import data_utils as du

production_failures = du.load_failures()
production_failures.head(10)

,uwi,formatted_uwi,well_name,field_operating_area,failure_start_date,failure_end_date,cause,cause_detail,cause_category,action_taken,failure_detail,failure_description,failure_system,failure_system_category,failure_comments,failure_type_category,cause_comments,failure_type
0,19G092704225W300,19G/09-27-042-25W3/00,NBRI WINTER HZ 19C10-27-15A9-27-42-25,Winter,2026-10-26,None,Fatigue,NaN,None,NaN,Rods Routine - Upset,NaN,NaN,None,NaN,None,NaN,Rods
1,100121303701W400,100/12-13-037-01W4/00,CALTEX_RES 100 MH PROVOST 12-13-37-1,Bodo East,2026-10-26,None,Fatigue,NaN,None,NaN,Rods Routine - Broken Corod body,NaN,NaN,None,NaN,None,NaN,Rods
2,192162703528W300,192/16-27-035-28W3/00,NBRI CACTUS LAKE V2U DD 6C9-27-1B16-27-35-28,Cactus Lake,2026-10-26,None,Corrosion,Rods,None,NaN,Rods Routine - Broken Corod body,NaN,NaN,None,Broken corod @ 268m,None,NaN,Rods
3,193082103528W300,193/08-21-035-28W3/00,NBRI CACTUS LAKE V2U DD 11A9-21-4D8-21-35-28,Cactus Lake,2026-10-26,None,Corrosion,Rods,None,NaN,Rods Routine - Broken Corod body,NaN,NaN,None,Broken corod @ 258m,None,NaN,Rods
4,104082403729W300,104/08-24-037-29W3/00,CALTEX 104 HZ COSINE LAKE 8-24-37-29,Cosine,2026-10-26,None,Fatigue,NaN,None,NaN,Bent/Broken Rotor,NaN,NaN,None,Broken rotor 3' from top,None,NaN,Pump
5,102141703528W300,102/14-17-035-28W3/00,SRL CACTUS LAKE DD 3C14-17-35-28W3 (28B14-17),Cactus Lake,2026-10-26,None,NaN,NaN,None,NaN,Hole,NaN,NaN,None,Hole in the upset on joint 88 rite above the s...,None,NaN,Tubing
6,102122103701W400,102/12-21-037-01W4/00,CALTEX_RES 102 HZ PROVOST 12-21-37-1,Bodo East,2026-10-25,None,Corrosion,Rods,None,NaN,Rod - Broken Corod Body,NaN,NaN,None,NaN,None,NaN,Rods
7,101011203628W300,101/01-12-036-28W3/00,SRL CACTUS LAKE DD 2C1-12-36-28 (3C1-12),Cactus Lake,2026-10-25,None,Fatigue,NaN,None,NaN,Rods Routine - Polished Rod,NaN,NaN,None,NaN,None,NaN,Rods
8,19E092704225W300,19E/09-27-042-25W3/00,NBRI WINTER HZ 11C10-27-14D9-27-42-25,Winter,2026-10-25,None,Fatigue,NaN,None,NaN,Rods Routine - Polished Rod,NaN,NaN,None,NaN,None,NaN,Rods
9,191142503328W300,191/14-25-033-28W3/00,NBRI ET AL COURT V1U DD 6B3-36-3C14-25-33-28,Court,2026-10-25,None,Fatigue,NaN,None,NaN,Parted,NaN,NaN,None,Top of joint 55 broken threads,None,NaN,Tubing


In [2]:
production_failures = production_failures.drop(columns=[
    "formatted_uwi", "well_name", "cause_category", "action_taken", "failure_description",
    "failure_system", "failure_system_category", "failure_type_category", "cause_comments"
    ])  # failure_comments is kept: it records the failure depth


In [3]:
# Parse failure depth (m) and rod number from failure_comments, and standardise the free-text
# failure_type and cause spellings (e.g. 'Rod', 'Parted rod' -> 'Rods').
production_failures = du.extract_failure_depth(production_failures)
production_failures = du.clean_failures(production_failures)
production_failures[['failure_comments', 'failure_depth_m', 'failure_rod_number']].dropna(subset=['failure_comments']).head(10)

failure_rod_number parsed for 142 comments (rod count from surface; not converted to metres).
failure_depth_m parsed for 433 of 1,599 comments (27.1%); 15.1% of all failures.
failure_type: 29 spellings -> 7 standard values
cause: 36 spellings -> 27 standard values


,failure_comments,failure_depth_m,failure_rod_number
2,Broken corod @ 268m,268.0,<NA>
3,Broken corod @ 258m,258.0,<NA>
4,Broken rotor 3' from top,<NA>,<NA>
5,Hole in the upset on joint 88 rite above the s...,<NA>,<NA>
9,Top of joint 55 broken threads,<NA>,<NA>
13,Camloc noturn was ran upside down,<NA>,<NA>
15,Rod Body on Rotor is broken,<NA>,<NA>
17,Broken corod @ 241m.,241.0,<NA>
18,Broken botttom pin,<NA>,<NA>
19,Broken corod @ 251m,251.0,<NA>


In [4]:
production_failures.columns

Index(['uwi', 'field_operating_area', 'failure_start_date', 'failure_end_date',
       'cause', 'cause_detail', 'failure_detail', 'failure_comments',
       'failure_type', 'failure_depth_m', 'failure_rod_number',
       'failure_type_std', 'cause_std'],
      dtype='str')

In [5]:
production_failures_for_encoding = production_failures.drop(columns=[
    "uwi", "field_operating_area", "failure_start_date", "failure_end_date", "failure_detail",
    "failure_comments", "failure_depth_m", "failure_rod_number", "failure_type_std", "cause_std"
    ])
production_failures_for_encoding.columns

Index(['cause', 'cause_detail', 'failure_type'], dtype='str')

In [6]:
# category dtype instead of pd.factorize: factorize turns missing values into -1, which a model
# would treat as a real (smallest) code. Categories keep missing values as NaN.
production_failures_encoded_df = production_failures_for_encoding.astype("category")
production_failures_encoded_df.dtypes

cause           category
cause_detail    category
failure_type    category
dtype: object

In [7]:
# replace the text columns with their categorical versions (concat would duplicate column names)
production_failures_combined = production_failures.assign(
    **{col: production_failures_encoded_df[col] for col in production_failures_encoded_df.columns})
production_failures_combined.head()

,uwi,field_operating_area,failure_start_date,failure_end_date,cause,cause_detail,failure_detail,failure_comments,failure_type,failure_depth_m,failure_rod_number,failure_type_std,cause_std
0,19G092704225W300,Winter,2026-10-26,None,Fatigue,NaN,Rods Routine - Upset,NaN,Rods,<NA>,<NA>,Rods,Fatigue
1,100121303701W400,Bodo East,2026-10-26,None,Fatigue,NaN,Rods Routine - Broken Corod body,NaN,Rods,<NA>,<NA>,Rods,Fatigue
2,192162703528W300,Cactus Lake,2026-10-26,None,Corrosion,Rods,Rods Routine - Broken Corod body,Broken corod @ 268m,Rods,268.0,<NA>,Rods,Corrosion
3,193082103528W300,Cactus Lake,2026-10-26,None,Corrosion,Rods,Rods Routine - Broken Corod body,Broken corod @ 258m,Rods,258.0,<NA>,Rods,Corrosion
4,104082403729W300,Cosine,2026-10-26,None,Fatigue,NaN,Bent/Broken Rotor,Broken rotor 3' from top,Pump,<NA>,<NA>,Pump,Fatigue
